## 🚨 **IMPORTANT UPDATE: DNF/Lapped Athlete Handling**

This notebook has been updated to properly handle **DNF (Did Not Finish)** and **lapped athletes** who have missing data in later segments of the race.

### **Key Improvements:**
- **DNF Detection**: Athletes with missing critical segment times are properly identified as DNF
- **Pack Assignment**: DNF athletes get pack ID = -1 (instead of appearing in lead packs)
- **Visual Indicators**: DNF status clearly shown in all analyses and visualizations
- **Timeline Analysis**: Shows exactly when athletes DNF'd during the race

### **Why This Matters:**
Without proper DNF handling, athletes who drop out early can appear to be in the lead pack at later checkpoints due to missing time data. This fix ensures accurate pack dynamics analysis.

### **New Features:**
- `explore_pack_composition(checkpoint, -1)` to view DNF athletes
- DNF progression timeline throughout the race
- Accurate pack statistics excluding DNF athletes

# Triathlon Pack Dynamics Analysis - Hamburg 2025

## Overview
This notebook analyzes race pack dynamics, particularly during the bike segment, to help triathletes and coaches understand:
- Pack formation and evolution throughout the race
- Strategic positioning opportunities
- Drafting group identification
- Gap analysis between groups
- Tactical insights for race strategy

### Pack Classification Logic
We define a "pack" as a group of athletes with elapsed times within 2 seconds of each other at any given split.
- If the gap between consecutive athletes > 2 seconds → new pack
- Athletes within the same pack have tactical/drafting opportunities
- Pack dynamics can change throughout the race segments

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Styling
plt.style.use('default')
sns.set_palette("husl")

## Data Loading and Exploration

In [2]:
# Load the Hamburg 2025 detailed results
file_path = '../data/Detailed results Hamburg 2025.xlsx'

# Read Excel file - may need to specify sheet name
try:
    # First, let's see what sheets are available
    excel_file = pd.ExcelFile(file_path)
    print("Available sheets:")
    for sheet in excel_file.sheet_names:
        print(f"- {sheet}")
    
    # Load the first sheet to start
    df = pd.read_excel(file_path, sheet_name=0)
    print(f"\nLoaded sheet: {excel_file.sheet_names[0]}")
    print(f"Shape: {df.shape}")
    
except Exception as e:
    print(f"Error loading file: {e}")

print("\nColumn names:")
print(df.columns.tolist())


Available sheets:
- Men
- Women

Loaded sheet: Men
Shape: (55, 37)

Column names:
['Gender', 'Rank', 'Bib', 'Name', 'Nat', 'S1', 'T1', 'B1T1', 'B1T2', 'BL1', 'B2T1', 'B2T2', 'BL2', 'B3T1', 'B3T2', 'BL3', 'B4T1', 'B4T2', 'BL4', 'B5T1', 'B5T2', 'BL5', 'B6T1', 'B6T2', 'BL6', 'T2', 'RT1', 'RL1', 'RT2', 'RL2', '  ', 'Swim', 'TA1', 'Bike', 'TA2', 'Run', 'Total']


## Data Preprocessing and Time Conversion

In [3]:
def convert_time_to_seconds(time_str):
    """
    Convert time string (HH:MM:SS or MM:SS) to total seconds
    """
    if pd.isna(time_str):
        return np.nan
    
    try:
        # Handle different time formats
        if isinstance(time_str, str):
            parts = time_str.split(':')
            if len(parts) == 3:  # HH:MM:SS
                hours, minutes, seconds = map(int, parts)
                return hours * 3600 + minutes * 60 + seconds
            elif len(parts) == 2:  # MM:SS
                minutes, seconds = map(int, parts)
                return minutes * 60 + seconds
        elif isinstance(time_str, (int, float)):
            return time_str  # Already in seconds
    except:
        return np.nan
    
    return np.nan

def seconds_to_time_str(seconds):
    """
    Convert seconds back to HH:MM:SS format
    """
    if pd.isna(seconds):
        return np.nan
    
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    
    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    else:
        return f"{minutes:02d}:{secs:02d}"

In [4]:
# Identify time columns (this will depend on the actual column structure)
# We'll need to adapt this based on what we see in the data
time_columns = [col for col in df.columns if any(keyword in col.lower() for keyword in 
                ['time', 'split', 'swim', 'bike', 'run', 't1', 't2', 'finish', 'bl', 'rl','s1'])]

print("Identified time columns:")
for col in time_columns:
    print(f"- {col}")


Identified time columns:
- S1
- T1
- B1T1
- B1T2
- BL1
- B2T1
- B2T2
- BL2
- B3T1
- B3T2
- BL3
- B4T1
- B4T2
- BL4
- B5T1
- B5T2
- BL5
- B6T1
- B6T2
- BL6
- T2
- RT1
- RL1
- RT2
- RL2
- Swim
- Bike
- Run


## Pack Dynamics Analysis Functions

In [5]:
def identify_packs(times, max_gap_to_leader=2, max_gap_within_pack=1):
    """
    Identify packs with two thresholds:
    - max_gap_within_pack: max allowed gap to previous athlete to stay in the pack
    - max_gap_to_leader: max allowed gap to the pack leader to stay in the pack

    Returns: array of pack ids (starting from 0), -1 for DNF/lapped (NaN)
    """
    import numpy as np
    import pandas as pd

    if len(times) == 0:
        return np.array([])

    pack_ids = np.full(len(times), -1, dtype=int)
    current_pack = 0

    valid_indices = [i for i, t in enumerate(times) if pd.notna(t)]
    if not valid_indices:
        return pack_ids

    i = 0
    while i < len(valid_indices):
        leader_idx = valid_indices[i]
        pack_ids[leader_idx] = current_pack
        pack_leader_time = times[leader_idx]
        j = i + 1
        while j < len(valid_indices):
            curr_idx = valid_indices[j]
            prev_idx = valid_indices[j-1]
            gap_to_prev = times[curr_idx] - times[prev_idx]
            gap_to_leader = times[curr_idx] - pack_leader_time
            if gap_to_prev <= max_gap_within_pack or gap_to_leader <= max_gap_to_leader:
                pack_ids[curr_idx] = current_pack
                j += 1
            else:
                break
        current_pack += 1
        i = j

    return pack_ids

def calculate_elapsed_times_fixed(df):
    """
    UPDATED: Calculate cumulative elapsed times with proper DNF/lapped athlete handling
    """
    df_elapsed = df.copy()
    
    # Convert all time columns to seconds
    time_cols = ['S1', 'T1', 'B1T1', 'B1T2', 'BL1', 'B2T1', 'B2T2', 'BL2', 
                 'B3T1', 'B3T2', 'BL3', 'B4T1', 'B4T2', 'BL4', 'B5T1', 'B5T2', 
                 'BL5', 'B6T1', 'B6T2', 'BL6', 'T2', 'RT1', 'RL1', 'RT2', 'RL2']
    
    # Convert times to seconds
    for col in time_cols:
        if col in df_elapsed.columns:
            df_elapsed[f'{col}_sec'] = df_elapsed[col].apply(convert_time_to_seconds)
    
    # Calculate key elapsed time checkpoints
    checkpoints = {}
    
    # After swim - if no swim time, athlete is DNF from start
    checkpoints['Elapsed_After_Swim'] = df_elapsed['S1_sec'].copy()
    
    # After T1
    if 'T1_sec' in df_elapsed.columns:
        elapsed_after_t1 = df_elapsed['S1_sec'] + df_elapsed['T1_sec']
        # If swim time exists but T1 doesn't, mark as DNF after swim
        elapsed_after_t1 = elapsed_after_t1.where(
            df_elapsed['S1_sec'].notna() & df_elapsed['T1_sec'].notna(),
            np.nan
        )
        checkpoints['Elapsed_After_T1'] = elapsed_after_t1
    else:
        checkpoints['Elapsed_After_T1'] = checkpoints['Elapsed_After_Swim'].copy()
    
    # After each bike lap (CORRECTED - include ALL bike segments + PROPER DNF detection)
    elapsed_so_far = checkpoints['Elapsed_After_T1'].copy()
    
    for lap_num in range(1, 7):  # Bike laps 1-6
        # For each lap, add: B{lap}T1 + B{lap}T2 + BL{lap}
        bike_segments = [f'B{lap_num}T1_sec', f'B{lap_num}T2_sec', f'BL{lap_num}_sec']
        
        # Check if athlete has ALL required data for this lap (not just ANY)
        lap_has_complete_data = pd.Series(True, index=df_elapsed.index)
        lap_elapsed_time = pd.Series(0.0, index=df_elapsed.index)
        
        # All segments in the lap must have data for the lap to be valid
        for seg in bike_segments:
            if seg in df_elapsed.columns:
                segment_valid = df_elapsed[seg].notna()
                lap_has_complete_data = lap_has_complete_data & segment_valid
                lap_elapsed_time = lap_elapsed_time + df_elapsed[seg].fillna(0)
        
        # Add lap time only if athlete has complete data AND was racing before this lap
        elapsed_so_far = elapsed_so_far + lap_elapsed_time
        
        # If athlete was racing before but doesn't have complete lap data, mark as DNF
        elapsed_so_far = elapsed_so_far.where(
            elapsed_so_far.isna() | lap_has_complete_data,
            np.nan
        )
        
        checkpoints[f'Elapsed_After_Bike_Lap_{lap_num}'] = elapsed_so_far.copy()
    
    # After T2
    if 'T2_sec' in df_elapsed.columns:
        elapsed_after_t2 = elapsed_so_far + df_elapsed['T2_sec']
        # If had bike data but no T2, mark as DNF
        elapsed_after_t2 = elapsed_after_t2.where(
            elapsed_so_far.isna() | df_elapsed['T2_sec'].notna(),
            np.nan
        )
        elapsed_so_far = elapsed_after_t2
        checkpoints['Elapsed_After_T2'] = elapsed_so_far.copy()
    
    # After each run segment
    run_segments = ['RT1_sec', 'RL1_sec', 'RT2_sec', 'RL2_sec']
    
    for i, seg in enumerate(run_segments):
        if seg in df_elapsed.columns:
            # Check if athlete has this run segment
            has_run_data = df_elapsed[seg].notna()
            elapsed_so_far = elapsed_so_far + df_elapsed[seg].fillna(0)
            
            # If had previous data but no run data, mark as DNF
            elapsed_so_far = elapsed_so_far.where(
                elapsed_so_far.isna() | has_run_data,
                np.nan
            )
            
            checkpoints[f'Elapsed_After_Run_Seg_{i+1}'] = elapsed_so_far.copy()
    
    # Add all checkpoint columns to dataframe
    for checkpoint, times in checkpoints.items():
        df_elapsed[checkpoint] = times
    
    return df_elapsed, checkpoints

def analyze_pack_evolution(df, gap_threshold=2):
    """
    Analyze pack dynamics across multiple checkpoints throughout the race
    UPDATED: Properly handles DNF/lapped athletes
    
    Parameters:
    df: dataframe with race results
    gap_threshold: gap threshold in seconds for pack identification
    
    Returns:
    pack_evolution_df: DataFrame with pack IDs for each checkpoint (-1 for DNF/lapped)
    checkpoint_stats: Dictionary with pack statistics for each checkpoint
    """
    # Calculate elapsed times (FIXED VERSION with DNF handling)
    df_elapsed, checkpoints = calculate_elapsed_times_fixed(df)
    
    pack_evolution = df_elapsed[['Name', 'Bib', 'Rank']].copy()
    checkpoint_stats = {}
    
    # Analyze packs at each checkpoint
    for checkpoint_name, elapsed_times in checkpoints.items():
        
        # Create DataFrame with original index preserved
        temp_data = df_elapsed[['Name']].copy()
        temp_data[f'{checkpoint_name}_elapsed'] = elapsed_times
        temp_data['original_index'] = temp_data.index
        
        # Only sort valid (non-NaN) times for pack identification
        valid_data = temp_data[temp_data[f'{checkpoint_name}_elapsed'].notna()].copy()
        
        if len(valid_data) == 0:
            # No valid times at this checkpoint
            pack_evolution[f'{checkpoint_name}_pack'] = -1
            checkpoint_stats[checkpoint_name] = pd.DataFrame()
            continue
        
        # Sort by elapsed time at this checkpoint, keeping track of original indices
        sorted_data = valid_data.sort_values(f'{checkpoint_name}_elapsed').reset_index(drop=True)
        
        # Identify packs on sorted data (only for valid times)
        pack_ids = identify_packs(sorted_data[f'{checkpoint_name}_elapsed'].values, gap_threshold, 1)
        sorted_data['pack_id'] = pack_ids
        
        # Map pack IDs back to original athlete order using original_index
        pack_mapping = dict(zip(sorted_data['original_index'], sorted_data['pack_id']))
        
        # Initialize all athletes as DNF (-1), then update those with valid times
        pack_evolution[f'{checkpoint_name}_pack'] = -1
        for orig_idx, pack_id in pack_mapping.items():
            pack_evolution.loc[orig_idx, f'{checkpoint_name}_pack'] = pack_id
        
        # Calculate statistics for this checkpoint using sorted data for accurate grouping
        # Only include athletes with valid pack IDs (not DNF)
        valid_pack_data = sorted_data[sorted_data['pack_id'] >= 0]
        
        if len(valid_pack_data) > 0:
            pack_stats = valid_pack_data.groupby('pack_id').agg({
                f'{checkpoint_name}_elapsed': ['count', 'min', 'max', 'mean']
            }).round(2)
            
            pack_stats.columns = ['pack_size', 'fastest_time', 'slowest_time', 'avg_time']
            pack_stats['time_spread'] = pack_stats['slowest_time'] - pack_stats['fastest_time']
        else:
            pack_stats = pd.DataFrame()
        
        checkpoint_stats[checkpoint_name] = pack_stats
    
    return pack_evolution, checkpoint_stats, df_elapsed

def analyze_pack_changes(pack_evolution_df, checkpoints_list):
    """
    Analyze how athletes move between packs across checkpoints
    
    Parameters:
    pack_evolution_df: DataFrame from analyze_pack_evolution
    checkpoints_list: List of checkpoint column names
    
    Returns:
    pack_movement_analysis: DataFrame showing pack changes for each athlete
    """
    movement_df = pack_evolution_df[['Name', 'Bib']].copy()
    
    for i in range(len(checkpoints_list) - 1):
        current_checkpoint = f'{checkpoints_list[i]}_pack'
        next_checkpoint = f'{checkpoints_list[i+1]}_pack'
        
        if current_checkpoint in pack_evolution_df.columns and next_checkpoint in pack_evolution_df.columns:
            pack_change = (pack_evolution_df[next_checkpoint] - pack_evolution_df[current_checkpoint])
            movement_df[f'Pack_Change_{checkpoints_list[i]}_to_{checkpoints_list[i+1]}'] = pack_change
    
    # Calculate total pack movements
    change_cols = [col for col in movement_df.columns if 'Pack_Change_' in col]
    movement_df['Total_Pack_Changes'] = movement_df[change_cols].abs().sum(axis=1)
    movement_df['Net_Pack_Movement'] = movement_df[change_cols].sum(axis=1)
    
    return movement_df

pack_evo, check_stats, df_times = analyze_pack_evolution(df, 2)

# Add DNF analysis summary
def analyze_dnf_patterns(pack_evolution_df, checkpoints_list):
    """
    Analyze DNF (Did Not Finish) patterns throughout the race
    """
    print(f"\n🚨 DNF/LAPPED ATHLETE ANALYSIS")
    print(f"{'='*50}")
    
    dnf_summary = {}
    
    for checkpoint in checkpoints_list:
        pack_col = f'{checkpoint}_pack'
        if pack_col in pack_evolution_df.columns:
            dnf_count = (pack_evolution_df[pack_col] == -1).sum()
            racing_count = (pack_evolution_df[pack_col] >= 0).sum()
            dnf_summary[checkpoint] = {
                'dnf_count': dnf_count,
                'racing_count': racing_count,
                'dnf_percentage': (dnf_count / len(pack_evolution_df)) * 100
            }
    
    print("DNF progression throughout race:")
    for checkpoint, stats in dnf_summary.items():
        checkpoint_name = checkpoint.replace('Elapsed_After_', '').replace('_', ' ')
        print(f"  {checkpoint_name:<20}: {stats['dnf_count']:2d} DNF, {stats['racing_count']:2d} racing ({stats['dnf_percentage']:.1f}% DNF)")
    
    # Show athletes who DNF'd and when
    print(f"\nDNF Timeline:")
    for i, (_, athlete) in enumerate(pack_evolution_df.iterrows()):
        name = athlete['Name']
        pack_cols = [col for col in pack_evolution_df.columns if '_pack' in col]
        
        # Find when athlete DNF'd
        dnf_checkpoint = None
        for col in pack_cols:
            if athlete[col] == -1:
                dnf_checkpoint = col.replace('_pack', '').replace('Elapsed_After_', '').replace('_', ' ')
                break
        
        if dnf_checkpoint:
            rank = athlete['Rank'] if 'Rank' in athlete else 'N/A'
            print(f"  {name:<25} DNF after {dnf_checkpoint:<15} (Final Rank: {rank})")
    
    return dnf_summary

## Visualization Functions

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown, Output, VBox

def plot_pack_timeline(analysis_df, time_col, title_suffix=""):
    """
    Create a timeline visualization showing pack formation
    """
    fig = go.Figure()
    
    # Get unique packs and assign colors
    unique_packs = sorted(analysis_df['pack_id'].unique())
    colors = px.colors.qualitative.Set3[:len(unique_packs)]
    
    for i, pack_id in enumerate(unique_packs):
        pack_data = analysis_df[analysis_df['pack_id'] == pack_id]
        
        fig.add_trace(go.Scatter(
            x=pack_data[f'{time_col}_seconds'],
            y=[pack_id] * len(pack_data),
            mode='markers',
            marker=dict(size=10, color=colors[i % len(colors)]),
            name=f'Pack {pack_id + 1} ({len(pack_data)} athletes)',
            text=pack_data.get('Name', pack_data.index),
            hovertemplate='<b>%{text}</b><br>Time: %{x}s<br>Pack: %{y}<extra></extra>'
        ))
    
    fig.update_layout(
        title=f'Pack Formation - {time_col} {title_suffix}',
        xaxis_title='Time (seconds)',
        yaxis_title='Pack ID',
        yaxis=dict(tickmode='linear', tick0=0, dtick=1),
        height=400
    )
    
    return fig

def plot_pack_evolution_timeline(pack_evolution_df, checkpoints_list, selected_athletes=None):
    """
    Create an interactive timeline showing how packs evolve throughout the race
    
    Parameters:
    pack_evolution_df: DataFrame from analyze_pack_evolution
    checkpoints_list: List of checkpoint names to display
    selected_athletes: List of athlete names to highlight (optional)
    """
    fig = go.Figure()
    
    # Get all athletes or selected ones
    athletes = selected_athletes if selected_athletes else pack_evolution_df['Name'].tolist()
    
    # Create color palette
    colors = px.colors.qualitative.Plotly
    
    for i, athlete in enumerate(athletes[:20]):  # Limit to 20 for readability
        athlete_data = pack_evolution_df[pack_evolution_df['Name'] == athlete]
        
        if len(athlete_data) > 0:
            y_values = []
            x_values = []
            
            for checkpoint in checkpoints_list:
                pack_col = f'{checkpoint}_pack'
                if pack_col in pack_evolution_df.columns:
                    pack_id = athlete_data[pack_col].iloc[0]
                    y_values.append(pack_id)
                    x_values.append(checkpoint.replace('Elapsed_After_', '').replace('_', ' '))
            
            fig.add_trace(go.Scatter(
                x=x_values,
                y=y_values,
                mode='lines+markers',
                name=athlete,
                line=dict(color=colors[i % len(colors)], width=2),
                marker=dict(size=8),
                hovertemplate=f'<b>{athlete}</b><br>Checkpoint: %{{x}}<br>Pack: %{{y}}<extra></extra>'
            ))
    
    fig.update_layout(
        title='Pack Evolution Throughout Race',
        xaxis_title='Race Checkpoint',
        yaxis_title='Pack ID',
        height=600,
        xaxis=dict(tickangle=45),
        showlegend=True
    )
    
    return fig

def plot_pack_sizes_evolution(checkpoint_stats, checkpoints_list):
    """
    Show how pack sizes change throughout the race
    """
    fig = go.Figure()
    
    checkpoint_names = [cp.replace('Elapsed_After_', '').replace('_', ' ') for cp in checkpoints_list]
    
    # Get maximum number of packs across all checkpoints
    max_packs = max([len(stats) for stats in checkpoint_stats.values()])
    
    for pack_id in range(max_packs):
        pack_sizes = []
        for checkpoint in checkpoints_list:
            if checkpoint in checkpoint_stats:
                stats = checkpoint_stats[checkpoint]
                if pack_id in stats.index:
                    pack_sizes.append(stats.loc[pack_id, 'pack_size'])
                else:
                    pack_sizes.append(0)
            else:
                pack_sizes.append(0)
        
        fig.add_trace(go.Scatter(
            x=checkpoint_names,
            y=pack_sizes,
            mode='lines+markers',
            name=f'Pack {pack_id + 1}',
            line=dict(width=3),
            marker=dict(size=8)
        ))
    
    fig.update_layout(
        title='Pack Size Evolution Throughout Race',
        xaxis_title='Race Checkpoint',
        yaxis_title='Pack Size (Number of Athletes)',
        height=500,
        xaxis=dict(tickangle=45)
    )
    
    return fig

def plot_athlete_pack_movement(pack_evolution_df, checkpoints_list, athlete_name):
    """
    Track a specific athlete's movement through packs
    """
    athlete_data = pack_evolution_df[pack_evolution_df['Name'] == athlete_name]
    
    if len(athlete_data) == 0:
        print(f"Athlete '{athlete_name}' not found")
        return None
    
    pack_positions = []
    checkpoint_names = []
    
    for checkpoint in checkpoints_list:
        pack_col = f'{checkpoint}_pack'
        if pack_col in pack_evolution_df.columns:
            pack_id = athlete_data[pack_col].iloc[0]
            pack_positions.append(pack_id + 1)  # Add 1 for display (Pack 1, 2, 3...)
            checkpoint_names.append(checkpoint.replace('Elapsed_After_', '').replace('_', ' '))
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=checkpoint_names,
        y=pack_positions,
        mode='lines+markers',
        name=athlete_name,
        line=dict(color='red', width=4),
        marker=dict(size=12, color='red'),
        hovertemplate=f'<b>{athlete_name}</b><br>Checkpoint: %{{x}}<br>Pack: %{{y}}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'Pack Movement Analysis - {athlete_name}',
        xaxis_title='Race Checkpoint',
        yaxis_title='Pack Number',
        height=400,
        xaxis=dict(tickangle=45),
        yaxis=dict(tickmode='linear', tick0=1, dtick=1)
    )
    
    return fig

def plot_time_gaps_evolution(df_elapsed, checkpoints_list, gap_threshold=2):
    """
    Show how time gaps to the leader evolve throughout the race
    Creates a scatter plot where:
    - X-axis: Race position (1st, 2nd, 3rd, etc.)
    - Y-axis: Time gap to leader in seconds
    - Each checkpoint is a different colored series
    """
    fig = go.Figure()
    
    # Color palette for different checkpoints
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
              '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    for i, checkpoint in enumerate(checkpoints_list):
        if checkpoint in df_elapsed.columns:
            # Get valid times and sort by elapsed time
            valid_data = df_elapsed[['Name', checkpoint]].dropna()
            valid_data = valid_data.sort_values(checkpoint)
            
            # Calculate gaps to leader (leader has 0 gap)
            leader_time = valid_data[checkpoint].iloc[0]
            gaps_to_leader = valid_data[checkpoint] - leader_time
            positions = list(range(1, len(gaps_to_leader) + 1))
            
            # Create hover text with athlete names
            hover_text = [f"<b>{name}</b><br>Position: {pos}<br>Gap to Leader: {gap:.1f}s" 
                         for name, pos, gap in zip(valid_data['Name'], positions, gaps_to_leader)]
            
            fig.add_trace(
                go.Scatter(
                    x=positions,
                    y=gaps_to_leader,
                    mode='markers',
                    name=checkpoint.replace('Elapsed_After_', '').replace('_', ' '),
                    marker=dict(
                        size=8,
                        color=colors[i % len(colors)],
                        opacity=0.7
                    ),
                    hovertemplate='%{text}<extra></extra>',
                    text=hover_text
                )
            )
    
    # Add pack threshold line for reference
    fig.add_hline(
        y=gap_threshold, 
        line_dash="dash", 
        line_color="orange",
        annotation_text=f"Pack Threshold ({gap_threshold}s)",
        annotation_position="bottom right"
    )
    
    fig.update_layout(
        title='Time Gaps to Leader Throughout Race<br><sub>Lower left = Leader, Upper right = Last place with biggest gap</sub>',
        xaxis_title='Race Position',
        yaxis_title='Time Gap to Leader (seconds)',
        height=600,
        hovermode='closest',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02
        )
    )
    
    # Customize axes
    fig.update_xaxes(
        tickmode='linear',
        tick0=1,
        dtick=2,  # Show every 2nd position
        range=[0.5, None]  # Start slightly before position 1
    )
    
    fig.update_yaxes(
        tickmode='linear',
        range=[0, None]  # Start at 0 (leader's gap)
    )
    
    return fig



def create_interactive_time_gaps_viz(df_elapsed, checkpoints_list, pack_evolution_df, gap_threshold=2):
    """
    Interactive time gaps visualization using established packs from pack_evolution_df.
    Prints lead pack members, their times, and the gap to the next pack.
    """
    output = Output()

    def show_gaps_at_checkpoint(checkpoint_index=0):
        output.clear_output()
        checkpoint = checkpoints_list[checkpoint_index]
        pack_col = f"{checkpoint}_pack"
        if checkpoint not in df_elapsed.columns or pack_col not in pack_evolution_df.columns:
            with output:
                print("Checkpoint or pack column not found.")
            return go.Figure()

        valid_data = df_elapsed[['Name', checkpoint]].dropna().sort_values(checkpoint)
        leader_time = valid_data[checkpoint].iloc[0]
        gaps_to_leader = valid_data[checkpoint] - leader_time
        positions = list(range(1, len(gaps_to_leader) + 1))

        # Merge with pack_evolution_df to get pack assignments for this checkpoint
        packs_at_checkpoint = pack_evolution_df[['Name', pack_col]]
        valid_data = valid_data.merge(packs_at_checkpoint, on='Name', how='left')

        # Find the leader's pack (the pack of the athlete in position 1)
        leader_pack = valid_data[pack_col].iloc[0]
        lead_pack_athletes = valid_data[valid_data[pack_col] == leader_pack]

        # Find the next pack (if any)
        next_pack_athletes = valid_data[valid_data[pack_col] != leader_pack]
        next_pack_number = None
        gap_to_next_pack = None
        if not next_pack_athletes.empty:
            next_pack_number = next_pack_athletes[pack_col].iloc[0]
            last_lead_pack_time = lead_pack_athletes[checkpoint].max()
            first_next_pack_time = next_pack_athletes[checkpoint].min()
            gap_to_next_pack = first_next_pack_time - last_lead_pack_time

        fig = go.Figure()
        # All athletes
        fig.add_trace(
            go.Scatter(
                x=positions,
                y=gaps_to_leader,
                mode='markers',
                marker=dict(size=12, color='#1f77b4', opacity=0.8, line=dict(width=2, color='darkblue')),
                text=valid_data['Name'].tolist(),
                hovertemplate='<b>%{text}</b><br>Position: %{x}<br>Gap to Leader: %{y:.1f}s<br><extra></extra>',
                name='Athletes'
            )
        )
        # Highlight the established lead pack
        if len(lead_pack_athletes) > 1:
            pack_positions = [positions[i] for i in lead_pack_athletes.index]
            pack_gaps = gaps_to_leader.iloc[lead_pack_athletes.index]
            fig.add_trace(
                go.Scatter(
                    x=pack_positions,
                    y=pack_gaps,
                    mode='markers',
                    marker=dict(size=15, color='red', opacity=0.6, symbol='circle-open', line=dict(width=3, color='red')),
                    text=lead_pack_athletes['Name'].tolist(),
                    hovertemplate='<b>%{text}</b> (In Lead Pack)<br>Position: %{x}<br>Gap to Leader: %{y:.1f}s<br><extra></extra>',
                    name='Lead Pack'
                )
            )
        checkpoint_name = checkpoint.replace('Elapsed_After_', '').replace('_', ' ')
        fig.update_layout(
            title=f'Time Gaps to Leader - {checkpoint_name}<br>'
                  f'<sub>Checkpoint {checkpoint_index + 1} of {len(checkpoints_list)} | '
                  f'Lead Pack: {len(lead_pack_athletes)} athletes (established packs)</sub>',
            xaxis_title='Race Position',
            yaxis_title='Time Gap to Leader (seconds)',
            height=600,
            hovermode='closest',
            showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        fig.update_xaxes(tickmode='linear', tick0=1, dtick=2, range=[0.5, max(positions) + 0.5])
        fig.update_yaxes(tickmode='linear', range=[0, max(gaps_to_leader) * 1.05])

        # Print summary info
        with output:
            print(f"\nLead Pack ({len(lead_pack_athletes)} athletes, Pack {leader_pack}):")
            for i, row in lead_pack_athletes.iterrows():
                print(f"  {row['Name']:<25}  Time: {row[checkpoint]:.1f}s  Gap: {row[checkpoint] - leader_time:.1f}s")
            if gap_to_next_pack is not None:
                print(f"\nGap to next pack (Pack {next_pack_number}): {gap_to_next_pack:.1f} seconds")
            else:
                print("\nNo next pack at this checkpoint.")

        return fig

    checkpoint_options = [(f"{i+1}. {cp.replace('Elapsed_After_', '').replace('_', ' ')}", i) 
                         for i, cp in enumerate(checkpoints_list)]
    @interact(
        checkpoint_index=Dropdown(
            options=checkpoint_options,
            value=0,
            description='Checkpoint:'
        )
    )
    def interactive_gaps_viz(checkpoint_index):
        return show_gaps_at_checkpoint(checkpoint_index)

    display(VBox([output]))

In [17]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import Dropdown, VBox, HBox, Output

def seconds_to_time_str(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h}:{m:02d}:{s:02d}"

def create_interactive_pack_composition_viz(pack_evolution_df, df_elapsed, checkpoint_stats, checkpoints_list):
    output = Output()

    def show_pack_composition(checkpoint, selected_pack):
        output.clear_output()
        with output:
            if checkpoint not in checkpoint_stats:
                print(f"Checkpoint {checkpoint} not found!")
                return

            print(f"📊 PACK COMPOSITION ANALYSIS")
            print(f"{'='*50}")
            print(f"Checkpoint: {checkpoint.replace('Elapsed_After_', '').replace('_', ' ')}")
            print(f"Selected Pack: {selected_pack + 1}")

            # Get pack statistics for this checkpoint
            stats = checkpoint_stats[checkpoint]
            print(f"\nOverall Statistics:")
            print(f"- Total Packs: {len(stats)}")
            print(f"- Largest Pack: {stats['pack_size'].max()} athletes")
            print(f"- Average Pack Size: {stats['pack_size'].mean():.1f} athletes")

            # Create main visualization (1 row, 2 columns)
            fig = make_subplots(
                rows=1, cols=2,
                subplot_titles=[
                    'Pack Sizes Overview',
                    'Athletes in Selected Pack'
                ],
                specs=[[{"type": "bar"}, {"type": "table"}]]
            )

            # 1. Pack sizes bar chart
            pack_ids = list(stats.index)
            pack_sizes = [stats.loc[i, 'pack_size'] for i in pack_ids]
            pack_labels = [f"Pack {i+1}" for i in pack_ids]
            colors = ['red' if i == selected_pack else 'lightblue' for i in pack_ids]

            fig.add_trace(
                go.Bar(
                    x=pack_labels,
                    y=pack_sizes,
                    marker_color=colors,
                    name="Pack Sizes",
                    text=pack_sizes,
                    textposition='auto'
                ),
                row=1, col=1
            )

            # 2. Get athletes in selected pack
            pack_col = f'{checkpoint}_pack'
            selected_pack_athletes = pack_evolution_df[pack_evolution_df[pack_col] == selected_pack]

            athlete_info = []
            if len(selected_pack_athletes) > 0:
                for _, athlete in selected_pack_athletes.iterrows():
                    name = athlete['Name']
                    if checkpoint in df_elapsed.columns:
                        time_sec = df_elapsed.loc[df_elapsed['Name'] == name, checkpoint].iloc[0]
                        if pd.notna(time_sec):
                            time_str = seconds_to_time_str(time_sec)
                            athlete_info.append([name, time_sec, time_str])

                # Sort by time in seconds
                athlete_info.sort(key=lambda x: x[1] if x[1] is not None else float('inf'))

            # Add table to subplot (1,2)
            if athlete_info:
                fig.add_trace(
                    go.Table(
                        header=dict(
                            values=['Athlete', 'Time at Checkpoint'],
                            fill_color='lightgray',
                            align='left'
                        ),
                        cells=dict(
                            values=[
                                [row[0] for row in athlete_info],
                                [row[2] for row in athlete_info]
                            ],
                            fill_color='white',
                            align='left'
                        )
                    ),
                    row=1, col=2
                )

            fig.update_layout(
                title=f'Pack Composition Analysis - {checkpoint.replace("Elapsed_After_", "").replace("_", " ")}',
                height=600,
                showlegend=False
            )

            fig.show()

            # Print detailed pack summary
            print(f"\n📋 PACK {selected_pack + 1} DETAILS:")
            print(f"{'='*30}")
            if selected_pack in stats.index:
                pack_stat = stats.loc[selected_pack]
                print(f"Pack Size: {pack_stat['pack_size']} athletes")
                print(f"Time Spread: {pack_stat['time_spread']:.1f} seconds")
                print(f"Fastest Time: {seconds_to_time_str(pack_stat['fastest_time'])}")
                print(f"Slowest Time: {seconds_to_time_str(pack_stat['slowest_time'])}")
                
            else:
                print("  No stats found for this pack.")

    # Widget logic
    def update_pack_dropdown(*args):
        checkpoint = checkpoint_dropdown.value
        stats = checkpoint_stats[checkpoint]
        pack_options = [(f"Pack {i+1}", i) for i in stats.index]
        pack_dropdown.options = pack_options
        pack_dropdown.value = stats.index[0] if len(stats.index) > 0 else 0

    checkpoint_options = [(cp.replace('Elapsed_After_', '').replace('_', ' '), cp) for cp in checkpoints_list]
    checkpoint_dropdown = Dropdown(options=checkpoint_options, value=checkpoints_list[0], description='Checkpoint:')
    stats = checkpoint_stats[checkpoints_list[0]]
    pack_options = [(f"Pack {i+1}", i) for i in stats.index]
    pack_dropdown = Dropdown(options=pack_options, value=stats.index[0], description='Pack:')

    checkpoint_dropdown.observe(update_pack_dropdown, names='value')

    def on_change(change):
        show_pack_composition(checkpoint_dropdown.value, pack_dropdown.value)

    checkpoint_dropdown.observe(lambda change: on_change(change), names='value')
    pack_dropdown.observe(lambda change: on_change(change), names='value')

    # Initial display
    show_pack_composition(checkpoint_dropdown.value, pack_dropdown.value)
    display(VBox([HBox([checkpoint_dropdown, pack_dropdown]), output]))

# Example usage (uncomment and adjust for your data):
# create_interactive_pack_composition_viz(pack_evolution_df, df_elapsed, checkpoint_stats, checkpoints_list)
    # Widget logic
    def update_pack_dropdown(*args):
        checkpoint = checkpoint_dropdown.value
        stats = checkpoint_stats[checkpoint]
        pack_options = [(f"Pack {i+1}", i) for i in stats.index]
        pack_dropdown.options = pack_options
        pack_dropdown.value = stats.index[0] if len(stats.index) > 0 else 0

    checkpoint_options = [(cp.replace('Elapsed_After_', '').replace('_', ' '), cp) for cp in checkpoints_list]
    checkpoint_dropdown = Dropdown(options=checkpoint_options, value=checkpoints_list[0], description='Checkpoint:')
    stats = checkpoint_stats[checkpoints_list[0]]
    pack_options = [(f"Pack {i+1}", i) for i in stats.index]
    pack_dropdown = Dropdown(options=pack_options, value=stats.index[0], description='Pack:')

    checkpoint_dropdown.observe(update_pack_dropdown, names='value')

    def on_change(change):
        show_pack_composition(checkpoint_dropdown.value, pack_dropdown.value)

    checkpoint_dropdown.observe(lambda change: on_change(change), names='value')
    pack_dropdown.observe(lambda change: on_change(change), names='value')

    # Initial display
    show_pack_composition(checkpoint_dropdown.value, pack_dropdown.value)
    display(VBox([HBox([checkpoint_dropdown, pack_dropdown]), output]))

# Example usage (uncomment and adjust for your data):
# create_interactive_pack_composition_viz(pack_evolution_df, df_elapsed, checkpoint_stats, checkpoints_list)

## Analysis Execution

Now let's run the pack dynamics analysis on the Hamburg 2025 data. We'll start by identifying the relevant split time columns and then analyze pack formation.

In [20]:
# 1. Calculate elapsed times and analyze pack evolution
pack_evolution_df, checkpoint_stats, df_elapsed = analyze_pack_evolution(df, 2)

# 2. Show available checkpoints
checkpoints_list = list(checkpoint_stats.keys())


In [9]:
# Create comprehensive pack evolution visualizations

print("Creating pack evolution visualizations...")

# 1. Pack Evolution Timeline (show top 10 athletes for clarity)
top_10_athletes = df.head(10)['Name'].tolist()
fig1 = plot_pack_evolution_timeline(pack_evolution_df, checkpoints_list, selected_athletes=top_10_athletes)
fig1.show()

# 2. Pack Size Evolution
#fig2 = plot_pack_sizes_evolution(checkpoint_stats, checkpoints_list)
#fig2.show()

# 3. Individual athlete tracking (example with race winner)
race_winner = df.iloc[0]['Name']
fig3 = plot_athlete_pack_movement(pack_evolution_df, checkpoints_list, race_winner)
if fig3:
    fig3.show()
    
# 4. Time gaps evolution
#fig4 = plot_time_gaps_evolution(df_elapsed, checkpoints_list[:5], gap_threshold=2)  # Show first 5 checkpoints
#fig4.show()

create_interactive_time_gaps_viz(df_elapsed, checkpoints_list, pack_evolution_df, gap_threshold=2)
create_interactive_pack_composition_viz(pack_evolution_df, df_elapsed, checkpoint_stats, checkpoints_list)

Creating pack evolution visualizations...


interactive(children=(Dropdown(description='Checkpoint:', options=(('1. Swim', 0), ('2. T1', 1), ('3. Bike Lap…

In [22]:
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown, VBox, Output

output = Output()

# Prepare checkpoint options
checkpoint_options = [(f"{i+1}. {cp.replace('Elapsed_After_', '').replace('_', ' ')}", i) 
                     for i, cp in enumerate(checkpoints_list)]

def show_gaps_at_checkpoint(checkpoint_index=0):
    output.clear_output()
    checkpoint = checkpoints_list[checkpoint_index]
    pack_col = f"{checkpoint}_pack"
    if checkpoint not in df_elapsed.columns or pack_col not in pack_evolution_df.columns:
        with output:
            print("Checkpoint or pack column not found.")
        return go.Figure()

    valid_data = df_elapsed[['Name', checkpoint]].dropna().sort_values(checkpoint)
    leader_time = valid_data[checkpoint].iloc[0]
    gaps_to_leader = valid_data[checkpoint] - leader_time
    positions = list(range(1, len(gaps_to_leader) + 1))

    # Merge with pack_evolution_df to get pack assignments for this checkpoint
    packs_at_checkpoint = pack_evolution_df[['Name', pack_col]]
    valid_data = valid_data.merge(packs_at_checkpoint, on='Name', how='left')

    # Find the leader's pack (the pack of the athlete in position 1)
    leader_pack = valid_data[pack_col].iloc[0]
    lead_pack_athletes = valid_data[valid_data[pack_col] == leader_pack]

    # Find the next pack (if any)
    next_pack_athletes = valid_data[valid_data[pack_col] != leader_pack]
    next_pack_number = None
    gap_to_next_pack = None
    if not next_pack_athletes.empty:
        next_pack_number = next_pack_athletes[pack_col].iloc[0]
        last_lead_pack_time = lead_pack_athletes[checkpoint].max()
        first_next_pack_time = next_pack_athletes[checkpoint].min()
        gap_to_next_pack = first_next_pack_time - last_lead_pack_time

    fig = go.Figure()
    # All athletes
    fig.add_trace(
        go.Scatter(
            x=positions,
            y=gaps_to_leader,
            mode='markers',
            marker=dict(size=12, color='#1f77b4', opacity=0.8, line=dict(width=2, color='darkblue')),
            text=valid_data['Name'].tolist(),
            hovertemplate='<b>%{text}</b><br>Position: %{x}<br>Gap to Leader: %{y:.1f}s<br><extra></extra>',
            name='Athletes'
        )
    )
    # Highlight the established lead pack
    if len(lead_pack_athletes) > 1:
        pack_positions = [positions[i] for i in lead_pack_athletes.index]
        pack_gaps = gaps_to_leader.iloc[lead_pack_athletes.index]
        fig.add_trace(
            go.Scatter(
                x=pack_positions,
                y=pack_gaps,
                mode='markers',
                marker=dict(size=15, color='red', opacity=0.6, symbol='circle-open', line=dict(width=3, color='red')),
                text=lead_pack_athletes['Name'].tolist(),
                hovertemplate='<b>%{text}</b> (In Lead Pack)<br>Position: %{x}<br>Gap to Leader: %{y:.1f}s<br><extra></extra>',
                name='Lead Pack'
            )
        )
    checkpoint_name = checkpoint.replace('Elapsed_After_', '').replace('_', ' ')
    fig.update_layout(
        title=f'Time Gaps to Leader - {checkpoint_name}<br>'
              f'<sub>Checkpoint {checkpoint_index + 1} of {len(checkpoints_list)} | '
              f'Lead Pack: {len(lead_pack_athletes)} athletes (established packs)</sub>',
        xaxis_title='Race Position',
        yaxis_title='Time Gap to Leader (seconds)',
        height=600,
        hovermode='closest',
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig.update_xaxes(tickmode='linear', tick0=1, dtick=2, range=[0.5, max(positions) + 0.5])
    fig.update_yaxes(tickmode='linear', range=[0, max(gaps_to_leader) * 1.05])

    # Print summary info
    with output:
        print(f"\nLead Pack ({len(lead_pack_athletes)} athletes, Pack {leader_pack}):")
        for i, row in lead_pack_athletes.iterrows():
            print(f"  {row['Name']:<25}  Time: {row[checkpoint]:.1f}s  Gap: {row[checkpoint] - leader_time:.1f}s")
        if gap_to_next_pack is not None:
            print(f"\nGap to next pack (Pack {next_pack_number}): {gap_to_next_pack:.1f} seconds")
        else:
            print("\nNo next pack at this checkpoint.")

    return fig

interact(
    show_gaps_at_checkpoint,
    checkpoint_index=Dropdown(
        options=checkpoint_options,
        value=0,
        description='Checkpoint:'
    )
)
display(VBox([output]))

interactive(children=(Dropdown(description='Checkpoint:', options=(('1. Swim', 0), ('2. T1', 1), ('3. Bike Lap…

In [10]:
# Interactive Pack Analysis - Customize Your Exploration

print("=== INTERACTIVE PACK DYNAMICS EXPLORATION ===")
print("\nCustomize the analysis below by changing the parameters:")

# CUSTOMIZABLE PARAMETERS
SELECTED_CHECKPOINT = "Elapsed_After_Bike_Lap_2"  # Change this to any checkpoint
GAP_THRESHOLD = 2  # Change this to adjust pack sensitivity (1-5 seconds recommended)
ATHLETE_TO_TRACK = "Matthew Hauser"  # Change to any athlete name from the race

print(f"\nCurrent Settings:")
print(f"- Analyzing checkpoint: {SELECTED_CHECKPOINT.replace('Elapsed_After_', '').replace('_', ' ')}")
print(f"- Pack gap threshold: {GAP_THRESHOLD} seconds")
print(f"- Tracking athlete: {ATHLETE_TO_TRACK}")

# Analysis for selected checkpoint
if SELECTED_CHECKPOINT in checkpoint_stats:
    print(f"\n--- PACK ANALYSIS AT {SELECTED_CHECKPOINT.replace('Elapsed_After_', '').replace('_', ' ')} ---")
    stats = checkpoint_stats[SELECTED_CHECKPOINT]
    print(f"Number of packs: {len(stats)}")
    print(f"Largest pack: {stats['pack_size'].max()} athletes")
    print(f"Average pack size: {stats['pack_size'].mean():.1f} athletes")
    print(f"Biggest time spread within a pack: {stats['time_spread'].max():.1f} seconds")
    
    print(f"\nPack breakdown:")
    for pack_id in stats.index:
        size = stats.loc[pack_id, 'pack_size']
        spread = stats.loc[pack_id, 'time_spread']
        print(f"  Pack {pack_id + 1}: {size} athletes, {spread:.1f}s spread")

# Show athletes who changed packs the most
print(f"\n--- MOST ACTIVE PACK MOVERS ---")
top_movers = movement_analysis.nlargest(5, 'Total_Pack_Changes')
for _, athlete in top_movers.iterrows():
    name = athlete['Name']
    changes = athlete['Total_Pack_Changes']
    net_movement = athlete['Net_Pack_Movement']
    direction = "moved forward" if net_movement < 0 else "moved backward" if net_movement > 0 else "stayed similar"
    print(f"  {name}: {changes} pack changes ({direction})")

# Track specific athlete
if ATHLETE_TO_TRACK in pack_evolution_df['Name'].values:
    print(f"\n--- TRACKING {ATHLETE_TO_TRACK} ---")
    athlete_data = pack_evolution_df[pack_evolution_df['Name'] == ATHLETE_TO_TRACK]
    pack_cols = [col for col in pack_evolution_df.columns if '_pack' in col]
    
    print("Pack positions throughout race:")
    for i, col in enumerate(pack_cols[:6]):  # Show first 6 checkpoints
        if col in athlete_data.columns:
            checkpoint_name = col.replace('_pack', '').replace('Elapsed_After_', '').replace('_', ' ')
            pack_num = athlete_data[col].iloc[0]
            if pack_num == -1:
                print(f"  {checkpoint_name:<15}: DNF/Lapped")
            else:
                print(f"  {checkpoint_name:<15}: Pack {pack_num + 1}")
else:
    print(f"\nAthlete '{ATHLETE_TO_TRACK}' not found. Available athletes:")
    print(df['Name'].head(10).tolist())

print(f"\n{'='*50}")
print("💡 TIP: Change the parameters above and re-run this cell to explore different scenarios!")
print("💡 TIP: Use the visualizations above to see the bigger picture of pack dynamics!")

=== INTERACTIVE PACK DYNAMICS EXPLORATION ===

Customize the analysis below by changing the parameters:

Current Settings:
- Analyzing checkpoint: Bike Lap 2
- Pack gap threshold: 2 seconds
- Tracking athlete: Matthew Hauser

--- PACK ANALYSIS AT Bike Lap 2 ---
Number of packs: 11
Largest pack: 12 athletes
Average pack size: 4.8 athletes
Biggest time spread within a pack: 5.0 seconds

Pack breakdown:
  Pack 1: 1 athletes, 0.0s spread
  Pack 2: 11 athletes, 3.0s spread
  Pack 3: 12 athletes, 4.0s spread
  Pack 4: 1 athletes, 0.0s spread
  Pack 5: 10 athletes, 2.0s spread
  Pack 6: 10 athletes, 5.0s spread
  Pack 7: 2 athletes, 0.0s spread
  Pack 8: 3 athletes, 1.0s spread
  Pack 9: 1 athletes, 0.0s spread
  Pack 10: 1 athletes, 0.0s spread
  Pack 11: 1 athletes, 0.0s spread

--- MOST ACTIVE PACK MOVERS ---
  James Edgar: 36 pack changes (moved backward)
  Harry Leleu: 33 pack changes (moved backward)
  Gergely Kiss: 33 pack changes (moved backward)
  Takumi Hojo: 33 pack changes (moved 

## Interactive Pack Composition Explorer

This section provides a dynamic visualization to explore the exact composition of any pack at any checkpoint. You can:

1. **Select a checkpoint** from the dropdown to see pack formation at that moment in the race
2. **Choose a specific pack** to see detailed athlete information
3. **View pack statistics** including time spreads and athlete rankings
4. **Analyze pack dynamics** with interactive charts and athlete tables

### Features:
- **Pack Size Overview**: Bar chart showing all pack sizes at selected checkpoint
- **Time Distribution**: Histogram of times within the selected pack  
- **Pack Timeline**: Visual representation of athlete positioning
- **Athlete Details**: Complete table with names, bibs, ranks, and times

## Advanced Pack Analysis Ideas

Here are additional analysis concepts we can implement once we understand the data structure:

### 1. Pack Evolution Analysis
- Track how packs form, split, and merge across different segments
- Identify athletes who consistently stay in lead packs vs. those who move between packs

### 2. Strategic Positioning Insights
- Analyze success rates of athletes starting in different pack positions
- Identify optimal pack sizes for different race strategies

### 3. Bike Segment Specific Analysis
- Drafting group identification (especially important in non-draft legal races)
- Gap analysis at key bike split points
- Power/speed implications of pack positioning

### 4. Transition Impact
- How T1 and T2 times affect pack positioning
- Athletes who use transitions strategically to change packs

### 5. Race Outcome Correlation
- Correlation between pack positioning at different splits and final results
- Identify critical race moments where pack position matters most

## Interactive Dashboard Ideas

Once we have the core analysis working, we can create interactive visualizations:

1. **Pack Timeline Slider**: Interactive timeline showing pack evolution throughout race
2. **Athlete Tracker**: Follow specific athletes through different packs
3. **Gap Threshold Adjuster**: Allow users to adjust the gap threshold and see how it affects pack identification
4. **Race Strategy Simulator**: Show implications of different tactical decisions

In [11]:
# Placeholder for interactive dashboard creation
# Will implement once we have the core analysis working

print("Interactive dashboard components will be added here")
print("This will include widgets for threshold adjustment and athlete tracking")

Interactive dashboard components will be added here
This will include widgets for threshold adjustment and athlete tracking


## Summary and Next Steps

This notebook provides a framework for comprehensive pack dynamics analysis. The key innovations include:

1. **Automated Pack Detection**: Using time gap thresholds to identify tactical groups
2. **Multi-segment Analysis**: Tracking pack evolution across swim, bike, and run
3. **Strategic Insights**: Identifying optimal positioning and tactical opportunities
4. **Visual Analytics**: Clear visualizations for coaches and athletes

### Potential Applications:
- **Race Strategy Planning**: Understanding when to stay with packs vs. break away
- **Training Focus**: Identifying skills needed for pack positioning
- **Performance Analysis**: Post-race tactical review
- **Competitive Intelligence**: Understanding competitor strategies

### Next Steps:
1. Load and analyze the Hamburg 2025 data
2. Refine pack detection algorithms based on actual race dynamics
3. Implement advanced analysis functions
4. Create interactive dashboard for real-time analysis
5. Validate insights with race video/expert knowledge

## 🔧 **Testing: Complete Lap Data Requirement Fix**

This section tests the fix for incomplete lap data handling. Previously, if any segment in a lap had data (e.g., B1T1), the lap was considered valid even if other segments (B1T2, BL1) were missing. 

### **The Fix:**
- **Before**: `lap_has_data = ANY segment has data` (OR logic)
- **After**: `lap_has_complete_data = ALL segments have data` (AND logic)

### **Test Case: Matthew Hauser**
Let's verify that Hauser is now correctly placed in pack analysis.

In [12]:
# Test the fix by checking raw data and pack assignment

print("🔧 TESTING: Complete Lap Data Requirement Fix")
print("="*60)

# Check Hauser's raw data first
print(f"🎯 MATTHEW HAUSER RAW DATA ANALYSIS:")
print("="*40)

hauser_original = df[df['Name'] == 'Matthew Hauser']

if len(hauser_original) > 0:
    print(f"Matthew Hauser's raw bike segment data:")
    bike_segments = ['B1T1', 'B1T2', 'BL1', 'B2T1', 'B2T2', 'BL2']
    
    print("Segment   | Value     | Status")
    print("-"*35)
    
    bike_lap_1_complete = True
    bike_lap_2_complete = True
    
    for i, seg in enumerate(bike_segments):
        if seg in hauser_original.columns:
            value = hauser_original[seg].iloc[0]
            if pd.isna(value):
                status = "❌ MISSING"
                if seg in ['B1T1', 'B1T2', 'BL1']:
                    bike_lap_1_complete = False
                elif seg in ['B2T1', 'B2T2', 'BL2']:
                    bike_lap_2_complete = False
            else:
                status = "✅ Present"
            print(f"{seg:<9} | {str(value):<9} | {status}")
    
    print(f"\nLap Completeness Analysis:")
    print(f"Bike Lap 1 Complete: {'✅ YES' if bike_lap_1_complete else '❌ NO'}")
    print(f"Bike Lap 2 Complete: {'✅ YES' if bike_lap_2_complete else '❌ NO'}")
    
    # Now check his pack assignments with current vs fixed logic
    print(f"\n🔍 PACK ASSIGNMENT COMPARISON:")
    print("="*35)
    
    # Get his pack assignment at Bike Lap 2 from existing analysis
    if 'Elapsed_After_Bike_Lap_2_pack' in pack_evolution_df.columns:
        hauser_pack_data = pack_evolution_df[pack_evolution_df['Name'] == 'Matthew Hauser']
        if len(hauser_pack_data) > 0:
            current_pack = hauser_pack_data['Elapsed_After_Bike_Lap_2_pack'].iloc[0]
            
            print(f"Current Pack Assignment (Bike Lap 2): ", end="")
            if current_pack == -1:
                print("DNF/Incomplete ✅")
            else:
                print(f"Pack {current_pack + 1} ❌")
                
            print(f"\nExpected Result:")
            if not bike_lap_2_complete:
                print("✅ Should be DNF/Incomplete due to missing B2T2 and BL2")
            else:
                print("✅ Should be in a valid pack")
    
else:
    print("❌ Matthew Hauser not found in data")

print(f"\n💡 ANALYSIS:")
print("If B2T2 and BL2 are missing, Hauser should be marked as DNF after Bike Lap 2")
print("This prevents him from appearing in incorrect packs for later checkpoints.")

🔧 TESTING: Complete Lap Data Requirement Fix
🎯 MATTHEW HAUSER RAW DATA ANALYSIS:
Matthew Hauser's raw bike segment data:
Segment   | Value     | Status
-----------------------------------
B1T1      | 00:00:57  | ✅ Present
B1T2      | 00:02:37  | ✅ Present
BL1       | 00:01:09  | ✅ Present
B2T1      | 00:00:52  | ✅ Present
B2T2      | 00:02:33  | ✅ Present
BL2       | 00:01:07  | ✅ Present

Lap Completeness Analysis:
Bike Lap 1 Complete: ✅ YES
Bike Lap 2 Complete: ✅ YES

🔍 PACK ASSIGNMENT COMPARISON:
Current Pack Assignment (Bike Lap 2): Pack 2 ❌

Expected Result:
✅ Should be in a valid pack

💡 ANALYSIS:
If B2T2 and BL2 are missing, Hauser should be marked as DNF after Bike Lap 2
This prevents him from appearing in incorrect packs for later checkpoints.


In [13]:
# Detailed analysis of Pack formation at Bike Lap 2

print("🔍 DETAILED PACK ANALYSIS AT BIKE LAP 2")
print("="*50)

# Check who's in each pack at Bike Lap 2
checkpoint_name = "Elapsed_After_Bike_Lap_2"

if checkpoint_name in checkpoint_stats:
    stats = checkpoint_stats[checkpoint_name]
    print(f"Pack breakdown at Bike Lap 2:")
    print(f"Total packs: {len(stats)}")
    
    for pack_id in sorted(stats.index):
        pack_col = f'{checkpoint_name}_pack'
        pack_athletes = pack_evolution_df[pack_evolution_df[pack_col] == pack_id]
        size = stats.loc[pack_id, 'pack_size']
        fastest = stats.loc[pack_id, 'fastest_time']
        slowest = stats.loc[pack_id, 'slowest_time']
        spread = stats.loc[pack_id, 'time_spread']
        
        print(f"\n📦 PACK {pack_id + 1}: {size} athletes")
        print(f"   Time range: {seconds_to_time_str(fastest)} - {seconds_to_time_str(slowest)} (spread: {spread:.1f}s)")
        print(f"   Athletes:")
        
        for i, (_, athlete) in enumerate(pack_athletes.iterrows()):
            name = athlete['Name']
            # Get their actual time at this checkpoint
            athlete_time = df_elapsed.loc[df_elapsed['Name'] == name, checkpoint_name]
            if len(athlete_time) > 0 and pd.notna(athlete_time.iloc[0]):
                time_str = seconds_to_time_str(athlete_time.iloc[0])
                print(f"     {i+1:2d}. {name:<25} ({time_str})")
            else:
                print(f"     {i+1:2d}. {name:<25} (No time)")

# Check for any athletes with incomplete data affecting pack formation
print(f"\n🚨 CHECKING FOR INCOMPLETE DATA ISSUES:")
print("="*45)

# Look for athletes who might have partial data causing pack assignment issues
incomplete_data_athletes = []

for i, (_, athlete) in enumerate(df.iterrows()):
    name = athlete['Name']
    
    # Check bike lap 2 segments
    b2_segments = ['B2T1', 'B2T2', 'BL2']
    b2_data = {}
    b2_complete = True
    
    for seg in b2_segments:
        if seg in athlete:
            value = athlete[seg]
            b2_data[seg] = value
            if pd.isna(value):
                b2_complete = False
        else:
            b2_data[seg] = "N/A"
            b2_complete = False
    
    # If bike lap 2 is incomplete, this athlete might be affecting pack formation
    if not b2_complete:
        incomplete_data_athletes.append({
            'name': name,
            'b2_data': b2_data,
            'b2_complete': b2_complete
        })

if incomplete_data_athletes:
    print(f"Found {len(incomplete_data_athletes)} athletes with incomplete Bike Lap 2 data:")
    for athlete in incomplete_data_athletes[:5]:  # Show first 5
        print(f"\n🔸 {athlete['name']}")
        for seg, value in athlete['b2_data'].items():
            status = "✅" if pd.notna(value) and value != "N/A" else "❌"
            print(f"   {seg}: {value} {status}")
else:
    print("✅ All athletes have complete Bike Lap 2 data")

# Check Matthew Hauser's specific time and position
print(f"\n🎯 MATTHEW HAUSER DETAILED ANALYSIS:")
print("="*40)

hauser_time = df_elapsed.loc[df_elapsed['Name'] == 'Matthew Hauser', checkpoint_name]
if len(hauser_time) > 0 and pd.notna(hauser_time.iloc[0]):
    hauser_elapsed = hauser_time.iloc[0]
    print(f"Hauser's time at Bike Lap 2: {seconds_to_time_str(hauser_elapsed)} ({hauser_elapsed:.1f}s)")
    
    # Show athletes around Hauser's time to understand pack formation
    all_times = df_elapsed[checkpoint_name].dropna().sort_values()
    hauser_position = (all_times <= hauser_elapsed).sum()
    
    print(f"Hauser's position: {hauser_position} out of {len(all_times)} athletes")
    
    # Show athletes within 10 seconds of Hauser
    time_window = 10  # seconds
    nearby_athletes = df_elapsed[
        (df_elapsed[checkpoint_name] >= hauser_elapsed - time_window) & 
        (df_elapsed[checkpoint_name] <= hauser_elapsed + time_window) &
        (df_elapsed[checkpoint_name].notna())
    ].sort_values(checkpoint_name)
    
    print(f"\nAthletes within {time_window}s of Hauser:")
    for _, athlete in nearby_athletes.iterrows():
        name = athlete['Name']
        time_val = athlete[checkpoint_name]
        gap = time_val - hauser_elapsed
        pack_id = pack_evolution_df.loc[pack_evolution_df['Name'] == name, f'{checkpoint_name}_pack'].iloc[0]
        
        marker = "👑" if name == "Matthew Hauser" else "  "
        print(f"{marker} {name:<25} {seconds_to_time_str(time_val)} ({gap:+5.1f}s) Pack {pack_id + 1}")
else:
    print("❌ Could not find Hauser's time at Bike Lap 2")

🔍 DETAILED PACK ANALYSIS AT BIKE LAP 2
Pack breakdown at Bike Lap 2:
Total packs: 11

📦 PACK 1: 1 athletes
   Time range: 0:18:17 - 0:18:17 (spread: 0.0s)
   Athletes:
      1. Henry Graf                (0:18:17)

📦 PACK 2: 11 athletes
   Time range: 0:18:20 - 0:18:23 (spread: 3.0s)
   Athletes:
      1. Matthew Hauser            (0:18:22)
      2. Vasco Vilaca              (0:18:20)
      3. Alessio Crociani          (0:18:21)
      4. Miguel Hidalgo            (0:18:20)
      5. Csongor Lehmann           (0:18:20)
      6. Max Stapley               (0:18:21)
      7. Márk Dévay                (0:18:21)
      8. Tjebbe Kaindl             (0:18:23)
      9. Chase McQueen             (0:18:23)
     10. Darr Smith                (0:18:20)
     11. Miguel Tiago Silva        (0:18:23)

📦 PACK 3: 12 athletes
   Time range: 0:18:31 - 0:18:35 (spread: 4.0s)
   Athletes:
      1. Ricardo Batista           (0:18:34)
      2. Dorian Coninx             (0:18:33)
      3. Connor Bentley           

In [14]:
# Debug test - check if the function works
print("🔧 DEBUGGING: Testing calculate_elapsed_times_fixed function")

try:
    test_df_elapsed, test_checkpoints = calculate_elapsed_times_fixed(df)
    print(f"✅ calculate_elapsed_times_fixed works - returned {len(test_checkpoints)} checkpoints")
    print(f"   Checkpoints: {list(test_checkpoints.keys())[:3]}...")
except Exception as e:
    print(f"❌ Error in calculate_elapsed_times_fixed: {e}")
    import traceback
    traceback.print_exc()

try:
    result = analyze_pack_evolution(df, 2)
    if result is None:
        print("❌ analyze_pack_evolution returned None")
    else:
        print("✅ analyze_pack_evolution works")
except Exception as e:
    print(f"❌ Error in analyze_pack_evolution: {e}")
    import traceback
    traceback.print_exc()

🔧 DEBUGGING: Testing calculate_elapsed_times_fixed function
✅ calculate_elapsed_times_fixed works - returned 13 checkpoints
   Checkpoints: ['Elapsed_After_Swim', 'Elapsed_After_T1', 'Elapsed_After_Bike_Lap_1']...
✅ analyze_pack_evolution works


In [15]:
# Debug version of analyze_pack_evolution to find the issue

def analyze_pack_evolution_debug(df, gap_threshold=2):
    """
    Debug version with detailed logging
    """
    print("🔧 DEBUG: Starting analyze_pack_evolution_debug")
    
    try:
        # Calculate elapsed times (FIXED VERSION with DNF handling)
        print("   Step 1: Calculating elapsed times...")
        df_elapsed, checkpoints = calculate_elapsed_times_fixed(df)
        print(f"   ✅ Got {len(checkpoints)} checkpoints")
        
        pack_evolution = df_elapsed[['Name', 'Bib', 'Rank']].copy()
        checkpoint_stats = {}
        
        print("   Step 2: Analyzing packs at each checkpoint...")
        
        # Analyze packs at each checkpoint
        for i, (checkpoint_name, elapsed_times) in enumerate(checkpoints.items()):
            print(f"   Processing checkpoint {i+1}/{len(checkpoints)}: {checkpoint_name}")
            
            # Create DataFrame with original index preserved
            temp_data = df_elapsed[['Name']].copy()
            temp_data[f'{checkpoint_name}_elapsed'] = elapsed_times
            temp_data['original_index'] = temp_data.index
            
            # Only sort valid (non-NaN) times for pack identification
            valid_data = temp_data[temp_data[f'{checkpoint_name}_elapsed'].notna()].copy()
            
            print(f"     Valid athletes: {len(valid_data)}/{len(temp_data)}")
            
            if len(valid_data) == 0:
                # No valid times at this checkpoint
                pack_evolution[f'{checkpoint_name}_pack'] = -1
                checkpoint_stats[checkpoint_name] = pd.DataFrame()
                print("     No valid times, marking all as DNF")
                continue
            
            # Sort by elapsed time at this checkpoint, keeping track of original indices
            sorted_data = valid_data.sort_values(f'{checkpoint_name}_elapsed').reset_index(drop=True)
            
            # Identify packs on sorted data (only for valid times)
            pack_ids = identify_packs(sorted_data[f'{checkpoint_name}_elapsed'].values, gap_threshold)
            sorted_data['pack_id'] = pack_ids
            
            # Map pack IDs back to original athlete order using original_index
            pack_mapping = dict(zip(sorted_data['original_index'], sorted_data['pack_id']))
            
            # Initialize all athletes as DNF (-1), then update those with valid times
            pack_evolution[f'{checkpoint_name}_pack'] = -1
            for orig_idx, pack_id in pack_mapping.items():
                pack_evolution.loc[orig_idx, f'{checkpoint_name}_pack'] = pack_id
            
            # Calculate statistics for this checkpoint using sorted data for accurate grouping
            # Only include athletes with valid pack IDs (not DNF)
            valid_pack_data = sorted_data[sorted_data['pack_id'] >= 0]
            
            if len(valid_pack_data) > 0:
                pack_stats = valid_pack_data.groupby('pack_id').agg({
                    f'{checkpoint_name}_elapsed': ['count', 'min', 'max', 'mean']
                }).round(2)
                
                pack_stats.columns = ['pack_size', 'fastest_time', 'slowest_time', 'avg_time']
                pack_stats['time_spread'] = pack_stats['slowest_time'] - pack_stats['fastest_time']
            else:
                pack_stats = pd.DataFrame()
            
            checkpoint_stats[checkpoint_name] = pack_stats
            print(f"     Created {len(pack_stats)} packs")
        
        print("   ✅ All checkpoints processed successfully")
        return pack_evolution, checkpoint_stats, df_elapsed
        
    except Exception as e:
        print(f"   ❌ Error in analyze_pack_evolution_debug: {e}")
        import traceback
        traceback.print_exc()
        return None

# Test the debug version
print("Testing debug version...")
result = analyze_pack_evolution_debug(df, 2)
if result is not None:
    pack_evo_debug, check_stats_debug, df_times_debug = result
    print(f"✅ Debug version successful: {len(check_stats_debug)} checkpoints")
else:
    print("❌ Debug version also failed")

Testing debug version...
🔧 DEBUG: Starting analyze_pack_evolution_debug
   Step 1: Calculating elapsed times...
   ✅ Got 13 checkpoints
   Step 2: Analyzing packs at each checkpoint...
   Processing checkpoint 1/13: Elapsed_After_Swim
     Valid athletes: 55/55
     Created 8 packs
   Processing checkpoint 2/13: Elapsed_After_T1
     Valid athletes: 55/55
     Created 9 packs
   Processing checkpoint 3/13: Elapsed_After_Bike_Lap_1
     Valid athletes: 55/55
     Created 10 packs
   Processing checkpoint 4/13: Elapsed_After_Bike_Lap_2
     Valid athletes: 53/55
     Created 11 packs
   Processing checkpoint 5/13: Elapsed_After_Bike_Lap_3
     Valid athletes: 53/55
     Created 10 packs
   Processing checkpoint 6/13: Elapsed_After_Bike_Lap_4
     Valid athletes: 51/55
     Created 5 packs
   Processing checkpoint 7/13: Elapsed_After_Bike_Lap_5
     Valid athletes: 50/55
     Created 8 packs
   Processing checkpoint 8/13: Elapsed_After_Bike_Lap_6
     Valid athletes: 49/55
     Created 6 